# Python 3.14.0 New Features Demonstration

This notebook demonstrates some of the key new features introduced in Python 3.14.0 (released October 7, 2025).

## 1. Template String Literals (t-strings)

Python 3.14 introduces template string literals (t-strings), a new string formatting syntax that provides a safer, more explicit alternative to f-strings. T-strings are denoted by a t prefix and use ${...} for interpolation instead of curly braces. Unlike f-strings, which produce a str object, t-strings resolve to a Template instance.

- **Explicit and Safe:** T-strings require explicit conversion and don't automatically call __str__() or __format__(), reducing security risks and unexpected behavior.
- **Template Syntax:** Uses ${expression} for interpolation, making it visually distinct from f-strings and aligned with common templating languages.
- **Deferred Evaluation:** Unlike f-strings which evaluate immediately, t-strings can be stored and evaluated later, making them useful for templates.

### Basic Syntax

In [3]:
# Traditional f-string
name = "Alice"
age = 30
print(f"Hello, {name}! You are {age} years old.")

# New t-string (Python 3.14+)
print(t"Hello, ${name}! You are ${age} years old.")

Hello, Alice! You are 30 years old.
Template(strings=('Hello, $', '! You are $', ' years old.'), interpolations=(Interpolation('Alice', 'name', None, ''), Interpolation(30, 'age', None, '')))


You process t-strings by iterating over their components, using attributes such as .strings, .interpolations, and .values for safe and customized handling.

### Simple Interpolation

In [9]:
# Basic variable interpolation
username = "bob"
message = t"Welcome, ${username}!"
print(message) 

# Expressions inside t-strings
x = 10
y = 20
result = t"The sum of ${x} and ${y} is ${x + y}"
print(result)  

Template(strings=('Welcome, $', '!'), interpolations=(Interpolation('bob', 'username', None, ''),))
Template(strings=('The sum of $', ' and $', ' is $', ''), interpolations=(Interpolation(10, 'x', None, ''), Interpolation(20, 'y', None, ''), Interpolation(30, 'x + y', None, '')))


### Safety and Explicit Conversion

In [ ]:
class User:
    def __init__(self, name):
        self.name = name
    
    def __str__(self):
        return f"User({self.name})"

user = User("Charlie")

# F-string automatically calls __str__
print(f"User: {user}")  

# T-string requires explicit conversion
# print(t"User: ${user}")  # This would raise an error!
print(t"User: ${str(user)}")  

User: User(Charlie)
Template(strings=('User: $', ''), interpolations=(Interpolation('User(Charlie)', 'str(user)', None, ''),))


### SQL Query Templates (Safer)

In [10]:
# T-strings for SQL templates (illustration)
table = "users"
column = "email"

# More explicit about what's being interpolated
query = t"SELECT ${column} FROM ${table} WHERE active = true"
print(query)

Template(strings=('SELECT $', ' FROM $', ' WHERE active = true'), interpolations=(Interpolation('email', 'column', None, ''), Interpolation('users', 'table', None, '')))


## 2. Deferred Evaluation of Annotations

Annotations are no longer evaluated immediately, improving performance and allowing forward references.

In [ ]:
from typing import get_annotations

# In Python 3.14, this works without issues even if SomeClass isn't defined yet
class Node:
    def __init__(self, value: int):
        self.value = value
        self.next: 'Node | None' = None  # Forward reference works seamlessly
    
    def set_next(self, node: 'Node') -> 'Node':
        """Set the next node in the linked list"""
        self.next = node
        return self

# Annotations are stored but not evaluated until needed
node1 = Node(1)
node2 = Node(2)
node1.set_next(node2)

print(f"Node 1 value: {node1.value}")
print(f"Node 1 next value: {node1.next.value}")
print(f"\nAnnotations are deferred and evaluated only when accessed:")
print(f"Node.__init__ annotations: {get_annotations(Node.__init__)}")

## 3. Subinterpreters (PEP 734)

Python 3.14 adds support for running code in separate subinterpreters, enabling true parallelism.

In [ ]:
import sys

# Check if subinterpreters are available
print(f"Python version: {sys.version}")
print(f"\nSubinterpreters demonstration:")

try:
    from concurrent.futures import InterpreterPoolExecutor
    
    def cpu_intensive_task(n):
        """Simulate CPU-intensive work"""
        total = 0
        for i in range(n):
            total += i ** 2
        return total
    
    # Use InterpreterPoolExecutor to run tasks in separate interpreters
    with InterpreterPoolExecutor(max_workers=4) as executor:
        tasks = [1000000, 2000000, 1500000, 1800000]
        results = list(executor.map(cpu_intensive_task, tasks))
        
        print(f"Tasks completed: {len(results)}")
        print(f"Sample result: {results[0]:,}")
        
except ImportError:
    print("InterpreterPoolExecutor not available in this Python version.")
    print("This feature requires Python 3.14+")

## 4. Improved Error Messages

Python 3.14 provides better suggestions when you make typos in keywords.

In [ ]:
# Example of improved error messages
print("In Python 3.14, if you type:")
print("  improt math")
print("\nYou'll get a helpful suggestion:")
print("  SyntaxError: invalid syntax. Did you mean 'import'?")
print("\nSimilarly for other typos:")
print("  'whlie' → suggests 'while'")
print("  'els' → suggests 'else'")
print("  'retrn' → suggests 'return'")

## 5. Zstandard Compression Support (PEP 784)

New built-in support for Zstandard compression algorithm.

In [ ]:
try:
    import compression.zstd as zstd
    
    # Sample data to compress
    data = b"Python 3.14.0 brings many exciting new features! " * 100
    
    # Compress data
    compressed = zstd.compress(data, level=3)
    
    # Decompress data
    decompressed = zstd.decompress(compressed)
    
    print(f"Original size: {len(data):,} bytes")
    print(f"Compressed size: {len(compressed):,} bytes")
    print(f"Compression ratio: {len(data)/len(compressed):.2f}x")
    print(f"Decompression successful: {data == decompressed}")
    
except ImportError:
    print("compression.zstd module not available in this Python version.")
    print("This feature requires Python 3.14+")
    
    # Fallback demo with standard library
    import zlib
    data = b"Python 3.14.0 brings many exciting new features! " * 100
    compressed = zlib.compress(data)
    print(f"\nUsing zlib as fallback:")
    print(f"Original size: {len(data):,} bytes")
    print(f"Compressed size: {len(compressed):,} bytes")
    print(f"Compression ratio: {len(data)/len(compressed):.2f}x")

## 6. Free-Threaded Python (No-GIL)

Python 3.14 officially supports free-threaded builds without the Global Interpreter Lock.

In [ ]:
import sys
import threading
import time

# Check if running in free-threaded mode
is_free_threaded = hasattr(sys, '_is_gil_enabled') and not sys._is_gil_enabled()

print(f"Free-threaded mode: {is_free_threaded}")
print(f"Python version: {sys.version}")

def cpu_bound_work(n):
    """CPU-intensive task"""
    result = 0
    for i in range(n):
        result += i ** 2
    return result

# Demonstrate threading (benefits more in free-threaded mode)
start_time = time.time()

threads = []
for i in range(4):
    t = threading.Thread(target=cpu_bound_work, args=(1000000,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

elapsed = time.time() - start_time

print(f"\nCompleted 4 CPU-bound tasks in {elapsed:.3f} seconds")
if is_free_threaded:
    print("Running in free-threaded mode - true parallelism achieved!")
else:
    print("Running with GIL - threads execute sequentially for CPU-bound tasks")

## 7. Incremental Garbage Collection

Python 3.14 introduces incremental garbage collection to reduce pause times.

In [ ]:
import gc
import time

# Check garbage collection stats
print("Garbage Collection Information:")
print(f"GC is enabled: {gc.isenabled()}")
print(f"GC thresholds: {gc.get_threshold()}")
print(f"GC counts: {gc.get_count()}")

# Create some objects to demonstrate GC
large_list = []
for i in range(10000):
    large_list.append({'id': i, 'data': [j for j in range(100)]})

print(f"\nCreated {len(large_list)} objects")

# Force garbage collection and time it
start = time.perf_counter()
collected = gc.collect()
duration = time.perf_counter() - start

print(f"Objects collected: {collected}")
print(f"GC duration: {duration*1000:.3f} ms")
print("\nIn Python 3.14, incremental GC reduces pause times for large heaps!")

## 8. REPL Improvements

Python 3.14 includes syntax highlighting and improved autocompletion in the interactive shell.

In [ ]:
print("New REPL Features in Python 3.14:")
print("\n1. Syntax Highlighting:")
print("   - Keywords, strings, and numbers are colorized in real-time")
print("   - Customizable color themes")
print("\n2. Import Autocompletion:")
print("   - Tab completion now works for module names")
print("   - Example: 'from coll<TAB>' suggests 'collections'")
print("\n3. Better History:")
print("   - Improved navigation through command history")
print("   - Persistent history across sessions")
print("\n4. Colorized Output:")
print("   - unittest now has colored output like pytest")
print("   - pdb includes syntax highlighting")

## Summary

Python 3.14.0 brings significant improvements:

### Performance
- **JIT Compiler**: 3-5% performance boost on x86-64 and AArch64
- **Free-threaded Python**: No-GIL builds for true parallelism
- **Incremental GC**: Reduced pause times for large applications

### Developer Experience
- **T-strings**: Custom string processing with familiar syntax
- **Better error messages**: Helpful suggestions for typos
- **REPL improvements**: Syntax highlighting and better completion

### New Features
- **Deferred annotations**: Better performance and forward references
- **Subinterpreters**: True parallelism within Python
- **Zstandard compression**: Modern compression algorithm support
- **Safe debugger interface**: Attach debuggers without stopping processes

These features make Python 3.14 faster, more developer-friendly, and better suited for modern multi-core systems!